# 데이터 전처리 (data_preprocessing)

정제 노트북(`data-cleaning.ipynb`)이 만든 `coupon_cleaning.csv`를 읽어, **모델이 바로 학습할 수 있는 형태**로 바꾼다.

각 단계는 정제 노트북과 같이 **의도 → 관찰 → 결정** 순서로 기록한다.

| 순서 | 단계 | 하는 일 |
|---|---|---|
| 1 | 데이터 읽기 | `coupon_cleaning.csv`를 읽고 정제 결과가 맞는지 확인 |
| 2 | 분할 | 사람 단위 8:2 분할(본 분할) + 행 단위 무작위 분할(비교용) |
| 3 | 파생 변수 | `match_freq` 생성 (제안받은 쿠폰 업종의 평소 방문 빈도) |
| 4 | 인코딩 | 순서형 정수 / 원핫 인코딩 |
| 5 | 저장 | 모델 노트북이 읽을 파일 저장 |

**왜 분할이 맨 앞인가.** 인코딩처럼 데이터에서 무언가를 "배우는" 작업(예: 범주 목록 정하기)을 전체 데이터로 하면, 평가용 데이터의 정보가 학습에 새어 들어간다(정보 누수). 그래서 먼저 나누고, 그 뒤의 작업은 **학습용 데이터를 기준으로** 한다.

> **참고.** 인코딩 방식을 바꾸려면 4단계 인코딩 설정 셀의 `ORDINAL_MAPS`, `NUMERIC_COLS`, `ONEHOT_COLS`만 고치면 된다.

## 1. 데이터 읽기

**의도.** 정제 결과 파일을 읽고, 정제 노트북의 마지막 출력과 같은지 확인한다.

**확인할 것.**
- 12,610행 × 23열
- 빈 문자열이 남아 있지 않음
- 정수 컬럼 6개(`temperature`, `has_children`, `direction_same`, `distance`, `Y`, `profile_id`)가 정수로 읽힘

`coupon_cleaning.csv`가 없으면 `data-cleaning.ipynb`를 먼저 전체 실행한다.

In [1]:
import pandas as pd
from pathlib import Path

# 정제 노트북이 저장한 파일
CLEAN_FILE = Path("coupon_cleaning.csv")

# keep_default_na=False: "unknown" 같은 문자열을 결측(NaN)으로 바꾸지 않고 그대로 읽는다.
# 숫자로만 된 컬럼은 pandas가 자동으로 정수로 읽는다.
df = pd.read_csv(CLEAN_FILE, keep_default_na=False, encoding="utf-8-sig")

print(f"[행·열 수] {df.shape}")

# 정제 결과가 맞는지 확인
INT_COLS = ["temperature", "has_children", "direction_same", "distance", "Y", "profile_id"]

assert df.shape == (12610, 23), "정제 결과와 크기가 다르다. data-cleaning.ipynb를 다시 실행할 것"
assert (df == "").sum().sum() == 0, "빈 문자열이 남아 있다"
for col in INT_COLS:
    assert pd.api.types.is_integer_dtype(df[col]), f"{col}이(가) 정수가 아니다"

print(f"응답자(프로필) 수: {df['profile_id'].nunique()}명")
print(f"전체 수락률: {df['Y'].mean() * 100:.1f}%")
print("\n[컬럼별 자료형]")
print(df.dtypes.to_string())

[행·열 수] (12610, 23)
응답자(프로필) 수: 567명
전체 수락률: 56.8%

[컬럼별 자료형]
destination             object
passanger               object
weather                 object
temperature              int64
time                    object
coupon                  object
expiration              object
gender                  object
age                     object
maritalStatus           object
has_children             int64
education               object
occupation              object
income                  object
Bar                     object
CoffeeHouse             object
CarryAway               object
RestaurantLessThan20    object
Restaurant20To50        object
distance                 int64
direction_same           int64
Y                        int64
profile_id               int64


## 2. 분할

### 2-1. 사람 단위 8:2 분할 (본 분할)

**의도.** 학습용과 평가용을 **사람 단위**로 나눈다. 한 사람의 행(약 22행)은 전부 학습용이거나 전부 평가용이다.

**왜 사람 단위인가.**
- 한 사람이 약 22행을 답했으므로, 행을 무작위로 나누면 같은 사람이 학습용과 평가용에 동시에 들어간다.
- 그러면 모델이 "이 사람은 원래 잘 받는 사람"이라는 것을 학습용에서 외워 평가용을 맞히게 되고, 점수가 부풀려진다.
- 우리 서비스는 **처음 보는 운전자**를 판단해야 하므로, 평가도 학습 때 못 본 사람으로 해야 한다.

**방법.** `GroupShuffleSplit`에 `profile_id`를 그룹으로 준다. 평가용 비율은 20%, 재현을 위해 `random_state=42`로 고정한다.

**이 평가용 20%는 마지막 최종 평가 때 한 번만 쓴다.** 모델 비교와 조정은 학습용 80% 안에서 5-fold 그룹 교차검증으로 한다(모델 노트북).

In [2]:
from sklearn.model_selection import GroupShuffleSplit

RANDOM_STATE = 42   # 누가 실행해도 같은 분할이 나오도록 고정
TEST_SIZE = 0.2     # 평가용 비율 (사람 수 기준)

# 설명 변수(X), 타깃(y), 그룹(groups)을 나눈다.
# profile_id는 분할에만 쓰고 모델 입력에는 넣지 않으므로 X에서 뺀다.
X = df.drop(columns=["Y", "profile_id"])
y = df["Y"]
groups = df["profile_id"]

# 사람(profile_id) 단위로 8:2 분할
gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()
groups_train, groups_test = groups.iloc[train_idx].copy(), groups.iloc[test_idx].copy()

# 같은 사람이 양쪽에 들어가지 않았는지 확인 (가장 중요한 확인)
overlap = set(groups_train) & set(groups_test)
assert len(overlap) == 0, f"학습용과 평가용에 동시에 있는 사람: {len(overlap)}명"

# -------------------출력-------------------
print(f"학습용: {len(X_train):>6}행, {groups_train.nunique()}명, 수락률 {y_train.mean() * 100:.1f}%")
print(f"평가용: {len(X_test):>6}행, {groups_test.nunique()}명, 수락률 {y_test.mean() * 100:.1f}%")
print(f"평가용 행 비율: {len(X_test) / len(X) * 100:.1f}%")
print(f"양쪽에 동시에 있는 사람: {len(overlap)}명")

학습용:  10070행, 453명, 수락률 56.0%
평가용:   2540행, 114명, 수락률 59.8%
평가용 행 비율: 20.1%
양쪽에 동시에 있는 사람: 0명


**관찰.**  
1. 사람 수 기준으로 20%를 떼므로 행 비율은 정확히 20%가 아닐 수 있다(사람마다 행 수가 조금씩 다름).
2. 학습용과 평가용의 수락률이 전체 수락률(56.8%)과 크게 다르지 않아야 한쪽에 수락을 잘하는 사람만 과하게 몰리지 않았다고 볼 수 있다.

실제 결과는 학습용 56.0%(453명), 평가용 59.8%(114명)로 3.8%p 차이가 난다. 평가용에 수락을 잘하는 사람이 조금 더 뽑힌 것으로, 사람 수가 114명뿐이라 생기는 흔들림이다.
그래서 최종 평가 때는 baselineB0(전부 수락으로 찍기)의 정확도도 평가용 기준으로 다시 재서(59.8%) 비교한다. 전체 기준 56.8%와 비교하면 모델이 실제보다 좋아 보인다.

### 2-2. 행 단위 무작위 분할 (비교용)

**의도.** "사람 단위로 나누지 않으면 점수가 얼마나 부풀려지는가"를 보여 주기 위한 비교용 분할을 하나 더 만든다.
이 분할은 모델 노트북에서 같은 모델의 점수를 두 분할로 비교할 때만 쓰고, 모델 선택이나 최종 평가에는 쓰지 않는다.

**방법.** 행을 무작위로 8:2로 나눈다. `stratify=y`로 양쪽의 수락률을 맞춘다.

In [3]:
from sklearn.model_selection import train_test_split

# 행 단위 무작위 분할 (비교용)
Xr_train, Xr_test, yr_train, yr_test, gr_train, gr_test = train_test_split(
    X, y, groups,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,          # 학습용·평가용 수락률을 같게 맞춤
)

# 같은 사람이 양쪽에 얼마나 들어갔는지 센다 (사람 단위 분할과의 차이)
shared_people = set(gr_train) & set(gr_test)
shared_rows = gr_test.isin(shared_people).mean() * 100

print(f"학습용: {len(Xr_train):>6}행, 수락률 {yr_train.mean() * 100:.1f}%")
print(f"평가용: {len(Xr_test):>6}행, 수락률 {yr_test.mean() * 100:.1f}%")
print(f"\n양쪽에 동시에 있는 사람: {len(shared_people)}명 / 전체 {groups.nunique()}명")
print(f"평가용 행 중 '학습 때 이미 본 사람'의 행: {shared_rows:.1f}%")

학습용:  10088행, 수락률 56.8%
평가용:   2522행, 수락률 56.7%

양쪽에 동시에 있는 사람: 561명 / 전체 567명
평가용 행 중 '학습 때 이미 본 사람'의 행: 100.0%


**관찰.** 무작위 분할에서는 거의 모든 사람이 학습용과 평가용에 동시에 들어간다.
즉 평가용 행 대부분이 "학습 때 이미 본 사람"의 행이다. 이 분할로 잰 점수는 처음 보는 운전자에 대한 성능이 아니다.

**결정.** 본 분할은 2-1의 사람 단위 분할이다. 2-2는 점수 비교용으로만 저장한다.

## 3. 파생 변수: `match_freq`

**의도.** "지금 제안받은 쿠폰의 업종을 평소에 얼마나 자주 가는가"를 한 칸으로 만든다.

**왜 만드나.**
- 원본에는 방문 빈도가 업종별로 5칸(`Bar`, `CoffeeHouse`, `CarryAway`, `RestaurantLessThan20`, `Restaurant20To50`) 있다.
- 그런데 수락 여부에 중요한 것은 그중 **제안받은 쿠폰의 업종** 칸 하나다. 이 칸은 Y와의 연관(Cramér's V 0.353)이 원본의 어떤 컬럼보다 강하다.
- 트리는 이 조합을 스스로 찾을 수도 있지만, 로지스틱 회귀는 찾지 못한다. 이미 아는 구조를 한 칸으로 넣어 주면 네 모델이 같은 출발선에서 비교된다.
- 원본 5칸은 "전반적인 외식 성향" 정보가 남아 있으므로 지우지 않는다.

**정보 누수 여부.** 각 행 안의 값만으로 만들고 다른 행의 통계를 쓰지 않으므로, 분할 뒤에 만들어도 앞에 만들어도 결과가 같다. 여기서는 순서를 지키기 위해 분할 뒤에 만든다.

**매핑표가 필요한 이유.** 쿠폰 값과 방문 빈도 컬럼 이름이 다르다. 예: 쿠폰 `Restaurant(<20)` → 컬럼 `RestaurantLessThan20`.

In [4]:
# 쿠폰 값 → 그 업종의 방문 빈도 컬럼 이름
COUPON_TO_FREQ_COL = {
    "Bar": "Bar",
    "Coffee House": "CoffeeHouse",
    "Carry out & Take away": "CarryAway",
    "Restaurant(<20)": "RestaurantLessThan20",
    "Restaurant(20-50)": "Restaurant20To50",
}

def add_match_freq(X):
    """각 행의 쿠폰 업종에 해당하는 방문 빈도 값을 match_freq 컬럼으로 추가한다."""
    X = X.copy()
    # 매핑표에 없는 쿠폰이 있으면 바로 멈춘다 (새 쿠폰 종류가 들어온 경우 대비)
    unknown_coupons = set(X["coupon"]) - set(COUPON_TO_FREQ_COL)
    assert not unknown_coupons, f"매핑표에 없는 쿠폰: {unknown_coupons}"

    freq_col_per_row = X["coupon"].map(COUPON_TO_FREQ_COL)
    # 행마다 자기 쿠폰 업종의 컬럼 값을 골라 온다
    X["match_freq"] = [X.at[i, col] for i, col in zip(X.index, freq_col_per_row)]
    return X

# 본 분할과 비교용 분할 모두에 적용
X_train = add_match_freq(X_train)
X_test = add_match_freq(X_test)
Xr_train = add_match_freq(Xr_train)
Xr_test = add_match_freq(Xr_test)

# 확인: 학습용에서 match_freq 값별 수락률
FREQ_ORDER = ["never", "less1", "1~3", "4~8", "gt8", "unknown"]
check = (
    pd.DataFrame({"match_freq": X_train["match_freq"], "Y": y_train})
    .groupby("match_freq")["Y"].agg(["size", "mean"])
    .reindex(FREQ_ORDER)
    .rename(columns={"size": "행 수", "mean": "수락률(%)"})
)
check["수락률(%)"] = (check["수락률(%)"] * 100).round(1)
print("[학습용] match_freq별 수락률")
print(check.to_string())

[학습용] match_freq별 수락률
             행 수  수락률(%)
match_freq              
never       1684    22.0
less1       2551    51.0
1~3         3047    65.6
4~8         1872    71.6
gt8          784    70.7
unknown      132    52.3


**관찰.** 학습용에서도 방문 빈도가 높을수록 수락률이 올라가는 패턴이 그대로 보여야 한다(never 약 20%대 → 4~8 약 70%대).

**한계.**
- `Carry out & Take away` 쿠폰에서는 방문 빈도와 상관없이 수락률이 67~75%로 평평해서, 이 변수가 거의 작동하지 않는다.
- 실제 서비스에서는 운전자의 평소 방문 습관을 얻을 수 있다는 전제가 필요하다.

## 4. 인코딩

**의도.** 모델은 글자를 읽지 못하므로 문자열 컬럼을 숫자로 바꾼다.

**기준.** 순서가 있고, 수락률도 그 순서를 따라 **한 방향으로** 움직이는 컬럼만 순서형 정수로 넣는다. 나머지는 원핫 인코딩한다.

| 방식 | 컬럼 | 이유 |
|---|---|---|
| 순서형 정수 | `age`, `expiration`, `distance` | 순서대로 수락률이 대체로 한 방향으로 움직임 (`distance`는 정제 단계에서 이미 0/1/2로 만들었으므로 그대로 씀) |
| 그대로 사용 | `temperature`, `has_children`, `direction_same` | 이미 정수 |
| 원핫 | `time` | 순서는 있지만 수락률이 낮에 높고 아침·밤에 낮은 산 모양 |
| 원핫 | `income` | 순서는 있지만 수락률이 오르내리며 방향이 없음 |
| 원핫 | 방문 빈도 5칸, `match_freq` | `unknown`을 never~gt8 순서 안에 놓을 수 없음 |
| 원핫 | `destination`, `passanger`, `weather`, `coupon`, `gender`, `maritalStatus`, `education`, `occupation` | 순서가 없는 명목형 |

**두 방식의 차이.**
- **순서형 정수**: `age`의 below21 → 0, 21 → 1, …, 50plus → 7처럼 순서대로 번호를 매긴다. 칸 하나로 끝난다.
- **원핫**: 값마다 칸을 따로 만들어 해당하면 1, 아니면 0을 넣는다. `time`이면 `time_7AM`, `time_10AM`, … 5칸이 생긴다.

**왜 이 기준인가.** 트리는 어느 방식이든 잘 처리하지만, 로지스틱 회귀는 순서형 정수를 "값이 커질수록 결과가 한 방향으로 변한다"로 읽는다.
산 모양인 `time`을 정수로 넣으면 로지스틱이 그 패턴을 잡지 못해 부당하게 불리해진다. 네 모델에 같은 입력을 주기로 했으므로 로지스틱도 공정하게 비교되는 방식을 고른다.

**정보 누수를 막는 방법.** 원핫으로 만들 범주 목록은 **학습용 데이터에서만** 정한다. 평가용에만 있는 범주가 있으면 그 행의 해당 칸들은 모두 0이 된다.

In [5]:
# ---- 인코딩 설정 ---------------------------------------------------------
# 순서형 정수로 바꿀 컬럼과 그 순서 (앞에 있을수록 작은 번호)
# time, income을 순서형으로 바꾸려면 여기에 추가하고 ONEHOT_COLS에서 빼면 된다.
# distance도 순서형이지만 정제 단계에서 이미 0/1/2 정수로 만들었으므로 아래 NUMERIC_COLS에서 그대로 쓴다.
ORDINAL_MAPS = {
    "age": ["below21", "21", "26", "31", "36", "41", "46", "50plus"],
    "expiration": ["2h", "1d"],
}

# 이미 정수라 그대로 쓰는 컬럼 (distance 포함)
NUMERIC_COLS = ["temperature", "has_children", "direction_same", "distance"]

# 원핫 인코딩할 컬럼
ONEHOT_COLS = [
    "time", "income",
    "Bar", "CoffeeHouse", "CarryAway", "RestaurantLessThan20", "Restaurant20To50", "match_freq",
    "destination", "passanger", "weather", "coupon", "gender", "maritalStatus", "education", "occupation",
]

# 모든 컬럼이 정확히 한 방식에 배정됐는지 확인 (빠지거나 겹치면 멈춤)
assigned = list(ORDINAL_MAPS) + NUMERIC_COLS + ONEHOT_COLS
assert sorted(assigned) == sorted(X_train.columns), (
    f"배정 누락: {set(X_train.columns) - set(assigned)}, 중복/없는 컬럼: {set(assigned) - set(X_train.columns)}"
)
assert len(assigned) == len(set(assigned)), "한 컬럼이 두 방식에 배정됨"
print(f"컬럼 배정 확인: 순서형 {len(ORDINAL_MAPS)}개, 그대로 {len(NUMERIC_COLS)}개, 원핫 {len(ONEHOT_COLS)}개")

컬럼 배정 확인: 순서형 2개, 그대로 4개, 원핫 16개


In [6]:
def fit_categories(X_train):
    """원핫 인코딩에 쓸 범주 목록을 학습용 데이터에서만 정한다."""
    return {col: sorted(X_train[col].unique()) for col in ONEHOT_COLS}

def encode(X, categories):
    """X를 숫자만 있는 표로 바꾼다. categories는 반드시 학습용에서 정한 목록을 넘긴다."""
    X = X.copy()

    # 1) 순서형 정수: 정해 둔 순서대로 0, 1, 2, ...
    for col, order in ORDINAL_MAPS.items():
        X[col] = X[col].map({value: i for i, value in enumerate(order)})
        assert X[col].notna().all(), f"{col}에 순서표에 없는 값이 있다"

    # 2) 원핫: 학습용 범주 목록으로 고정한 뒤 칸을 만든다.
    #    학습용에 없던 값은 범주 밖(NaN)이 되어, 그 행의 해당 칸들이 모두 0이 된다.
    for col in ONEHOT_COLS:
        X[col] = pd.Categorical(X[col], categories=categories[col])
    X = pd.get_dummies(X, columns=ONEHOT_COLS, dtype=int)

    return X

# 본 분할: 학습용에서 범주 목록을 정하고, 학습용·평가용 모두 같은 목록으로 변환
categories = fit_categories(X_train)
X_train_enc = encode(X_train, categories)
X_test_enc = encode(X_test, categories)

# 비교용 분할: 같은 규칙으로 따로 처리 (자기 학습용에서 범주 목록을 정함)
categories_r = fit_categories(Xr_train)
Xr_train_enc = encode(Xr_train, categories_r)
Xr_test_enc = encode(Xr_test, categories_r)

# 확인
assert list(X_train_enc.columns) == list(X_test_enc.columns), "학습용과 평가용의 칸이 다르다"
assert X_train_enc.isna().sum().sum() == 0 and X_test_enc.isna().sum().sum() == 0, "결측이 생겼다"
assert all(pd.api.types.is_numeric_dtype(t) for t in X_train_enc.dtypes), "숫자가 아닌 칸이 남아 있다"

# 평가용에만 있는 범주 (있으면 해당 행은 그 컬럼의 원핫 칸이 모두 0)
unseen = {col: sorted(set(X_test[col]) - set(categories[col])) for col in ONEHOT_COLS}
unseen = {col: vals for col, vals in unseen.items() if vals}

print(f"인코딩 전 컬럼 수: {X_train.shape[1]}  →  인코딩 후 칸 수: {X_train_enc.shape[1]}")
print(f"학습용 {X_train_enc.shape}, 평가용 {X_test_enc.shape}")
print(f"평가용에만 있는 범주: {unseen if unseen else '없음'}")

인코딩 전 컬럼 수: 22  →  인코딩 후 칸 수: 109
학습용 (10070, 109), 평가용 (2540, 109)
평가용에만 있는 범주: 없음


In [7]:
# 인코딩 결과 미리보기: 컬럼별로 몇 칸이 되었는지
summary = []
for col in ORDINAL_MAPS:
    summary.append({"컬럼": col, "방식": "순서형 정수", "칸 수": 1})
for col in NUMERIC_COLS:
    label = "순서형 정수 (정제 단계에서 생성)" if col == "distance" else "그대로"
    summary.append({"컬럼": col, "방식": label, "칸 수": 1})
for col in ONEHOT_COLS:
    summary.append({"컬럼": col, "방식": "원핫", "칸 수": len(categories[col])})

summary = pd.DataFrame(summary).set_index("컬럼")
print(summary.to_string())
print(f"\n합계: {summary['칸 수'].sum()}칸")

# 첫 행이 어떻게 바뀌었는지 (0이 아닌 칸만)
first = X_train_enc.iloc[0]
print("\n[학습용 첫 행에서 0이 아닌 칸]")
print(first[first != 0].to_string())

                                       방식  칸 수
컬럼                                            
age                                순서형 정수    1
expiration                         순서형 정수    1
temperature                           그대로    1
has_children                          그대로    1
direction_same                        그대로    1
distance              순서형 정수 (정제 단계에서 생성)    1
time                                   원핫    5
income                                 원핫    9
Bar                                    원핫    6
CoffeeHouse                            원핫    6
CarryAway                              원핫    6
RestaurantLessThan20                   원핫    6
Restaurant20To50                       원핫    6
match_freq                             원핫    6
destination                            원핫    3
passanger                              원핫    4
weather                                원핫    3
coupon                                 원핫    5
gender                                 원핫    2
maritalStatus

**관찰.** 학습용과 평가용의 칸 이름과 순서가 완전히 같아야 하고, 결측이나 문자열이 남아 있지 않아야 한다.
원핫 칸이 많아 보이지만 대부분 0이고, 행마다 원핫 컬럼 하나당 정확히 한 칸만 1이다.

**한계.**
- `age`는 41·46세에서 수락률이 조금 오른다. 완전히 한 방향은 아니지만 대체로 감소하므로 순서형으로 두었다.
- 순서형 정수는 간격이 같다고 가정한다(21세와 26세의 차이 = 46세와 50세 이상의 차이). 실제 간격은 다르다.
- 응답자가 적은 범주(예: `occupation`의 일부 직업)는 원핫 칸이 소수의 사람만 가리킨다. 사람 단위 분할과 모델의 잎 크기 제한으로 외우기를 막고, 모델 비교 뒤 과적합 신호가 보이면 범주 묶기를 다시 검토한다.

## 5. 저장

**의도.** 모델 노트북이 이 결과를 그대로 읽을 수 있도록 파일로 저장한다.

**저장 파일** (`prep/` 폴더)

| 파일 | 내용 | 용도 |
|---|---|---|
| `group_train.csv` | 사람 단위 분할의 학습용 (인코딩 후) | 모델 비교, 교차검증, 조정 |
| `group_test.csv` | 사람 단위 분할의 평가용 (인코딩 후) | 마지막 최종 평가 **한 번만** |
| `random_train.csv`, `random_test.csv` | 행 단위 무작위 분할 (인코딩 후) | 점수 부풀림 비교용 |

각 파일에는 인코딩된 설명 변수와 함께 `Y`와 `profile_id`가 들어 있다.
- `profile_id`는 교차검증에서 사람 단위로 조각을 나눌 때 쓴다. **모델 입력에서는 반드시 뺀다.**
- `.gitignore`의 `*.csv` 규칙으로 커밋되지 않는다. 팀원은 정제 노트북 → 이 노트북 순서로 전체 실행해 재현한다.

In [8]:
PREP_DIR = Path("prep")
PREP_DIR.mkdir(exist_ok=True)

def save(X_enc, y, groups, name):
    """인코딩된 X에 Y와 profile_id를 붙여 저장한다."""
    out = X_enc.copy()
    out["Y"] = y.values
    out["profile_id"] = groups.values
    path = PREP_DIR / f"{name}.csv"
    out.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"저장: {path}  ({out.shape[0]}행 × {out.shape[1]}열)")

save(X_train_enc, y_train, groups_train, "group_train")
save(X_test_enc, y_test, groups_test, "group_test")
save(Xr_train_enc, yr_train, gr_train, "random_train")
save(Xr_test_enc, yr_test, gr_test, "random_test")

저장: prep\group_train.csv  (10070행 × 111열)
저장: prep\group_test.csv  (2540행 × 111열)
저장: prep\random_train.csv  (10088행 × 111열)
저장: prep\random_test.csv  (2522행 × 111열)


## 정리

| 항목 | 결과 |
|---|---|
| 본 분할 | 사람 단위 8:2, 같은 사람이 양쪽에 없음 |
| 비교용 분할 | 행 단위 무작위 8:2, 거의 모든 사람이 양쪽에 있음 |
| 파생 변수 | `match_freq` 1개 |
| 인코딩 | 순서형 2개, 그대로 4개, 원핫 16개 컬럼 |

**다음 단계 (모델 노트북).** `prep/group_train.csv`에서 `profile_id`로 5-fold 그룹 교차검증을 하며 기준선 → 로지스틱 회귀 → 의사결정나무 → 랜덤포레스트를 비교한다.
같은 모델을 `random_*` 파일로도 돌려 점수가 얼마나 부풀려지는지 확인한다. `group_test.csv`는 모델을 최종 선택한 뒤 한 번만 쓴다.